In [1]:
# imports

import os
from IPython.display import display, Video
import ipywidgets as widgets
import torch
import numpy as np
from helper import Model0, face_detect_trace, fft_transform

In [2]:
model = Model0()
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

Model0(
  (spatial_net): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=1e

In [3]:
video_path = "temp_vid.mp4"
op = widgets.Output()

def predict():
    with op:
        status.value = "Preprocessing..."
        faces = face_detect_trace(video_path)

        if faces is None or len(faces) < 10:
            print("No faces detected")
            return
            
        freqs = [fft_transform(face) for face in faces]

    faces = np.array(faces)
    freqs = np.array(freqs)
    
    spatial = torch.tensor(faces).permute(0, 3, 1, 2).float() / 255.0
    frequency = torch.tensor(freqs).unsqueeze(1).float() / 255.0

    spatial = spatial.unsqueeze(0)
    frequency = frequency.unsqueeze(0)

    status.value = "Predicting..."
    model.eval()

    with torch.no_grad():
        output = model(spatial, frequency)
        prob = torch.sigmoid(output).item()

    with op:
        status.value = "Result"
        display(Video(video_path))
        print(f"Prediction: {'FAKE' if prob > 0.5 else 'REAL'}")

    os.remove(video_path)
    
def temp_save(b):
    with op:
        op.clear_output()
        if uploader.value:
            status.value = "Saving Files..."

            info = list(uploader.value.values())[0] if isinstance(uploader.value, dict) else uploader.value[0]

            video_name = info["name"]
            video_bytes = info["content"]

            with open(video_path, "wb") as f:
                f.write(video_bytes)

            predict()

            uploader.value = ()
        else:
            with op:
                print("Upload Error")


status = widgets.Label(value="Upload a video")
uploader = widgets.FileUpload(accept='video/*', multiple=False)
button = widgets.Button(description = "Run Prediction")


button.on_click(temp_save)

display(status, uploader, button, op)

Label(value='Upload a video')

FileUpload(value=(), accept='video/*', description='Upload')

Button(description='Run Prediction', style=ButtonStyle())

Output()